# PPO Theory for the GOLDS Project

**Proximal Policy Optimization — from first principles to GOLDS hyperparameters**

This notebook walks through the math and intuition behind PPO, the algorithm that powers every agent in the GOLDS (Games On Learning Data Structures) project. By the end you will understand:

1. Why policy gradients work (and why vanilla REINFORCE is too noisy to be practical)
2. How PPO stabilizes training with a clipped surrogate objective
3. How Generalized Advantage Estimation (GAE) balances bias and variance
4. How the full PPO loss is assembled and what every GOLDS config knob controls
5. What the CnnPolicy network looks like and how many parameters it has

**Prerequisites:** You should be comfortable with basic calculus, probability, and have seen gradient descent in a supervised-learning context (e.g., training a classifier with cross-entropy loss). No prior RL knowledge is assumed.

In [ ]:
# Imports used throughout the notebook
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "font.size": 12,
})

---
## 1. Policy Gradient Foundations

### 1.1 The RL Objective

A **policy** $\pi_\theta(a \mid s)$ is a neural network parameterized by $\theta$ that outputs the probability of taking action $a$ in state $s$.

Our goal is to maximize the **expected cumulative discounted reward**:

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r_t\right]
$$

where $\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \ldots)$ is a trajectory sampled by running $\pi_\theta$ in the environment, and $\gamma \in [0, 1)$ is the **discount factor** (set to 0.99 in GOLDS).

### 1.2 The Policy Gradient Theorem

We cannot back-propagate through the environment, but the **policy gradient theorem** (Sutton et al., 2000) gives us an unbiased gradient estimator:

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot G_t\right]
$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the **return** from time $t$ onward.

### 1.3 REINFORCE — Monte Carlo Policy Gradient

The simplest algorithm based on this theorem is **REINFORCE** (Williams, 1992):

1. Collect a full trajectory $\tau$ by running $\pi_\theta$.
2. For each timestep $t$, compute $G_t$.
3. Update: $\theta \leftarrow \theta + \alpha \sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot G_t$

**Problem:** The return $G_t$ has extremely high variance because it depends on every future random action and transition. This makes learning slow and unstable.

### 1.4 Baselines and the Advantage Function

We can subtract any function $b(s_t)$ that does not depend on $a_t$ from $G_t$ **without** introducing bias:

$$
\nabla_\theta J(\theta) = \mathbb{E}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot \left(G_t - b(s_t)\right)\right]
$$

The optimal baseline (in the variance-minimizing sense) is close to the **state-value function**:

$$
V^\pi(s) = \mathbb{E}_{\tau \sim \pi}\left[G_t \mid s_t = s\right]
$$

This leads us to the **advantage function**:

$$
A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s)
$$

where $Q^\pi(s, a) = \mathbb{E}[G_t \mid s_t = s, a_t = a]$ is the action-value function.

**Intuition:** The advantage tells us *how much better* action $a$ is compared to what we *typically* do in state $s$. If $A > 0$, the action was better than average — increase its probability. If $A < 0$, decrease it. This signal has much lower variance than the raw return.

---
## 2. Trust Regions and PPO

### 2.1 The Problem with Large Updates

Policy gradient methods are **on-policy**: the data we collect is only valid for the current policy $\pi_\theta$. If we take a large gradient step, the new policy $\pi_{\theta'}$ may be very different, and the old data becomes misleading. This can cause training to collapse.

### 2.2 TRPO: Trust Region Policy Optimization

TRPO (Schulman et al., 2015) addresses this by solving a **constrained** optimization problem. Let $\theta_{\text{old}}$ be the parameters before the update. Define the **probability ratio**:

$$
r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}
$$

TRPO maximizes a surrogate objective subject to a KL-divergence constraint:

$$
\max_\theta \; \mathbb{E}_t\left[r_t(\theta) \, \hat{A}_t\right] \quad \text{s.t.} \quad \mathbb{E}_t\left[D_{\text{KL}}\left(\pi_{\theta_{\text{old}}}(\cdot \mid s_t) \;\|\; \pi_\theta(\cdot \mid s_t)\right)\right] \le \delta
$$

This guarantees monotonic improvement in theory, but requires computing second-order derivatives (the Fisher information matrix), which is expensive.

### 2.3 PPO: A Simpler Alternative

**Proximal Policy Optimization** (Schulman et al., 2017) replaces the hard KL constraint with a **clipped surrogate objective** that is trivial to implement with first-order optimizers like Adam.

The clipped objective is:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)\,\hat{A}_t, \;\text{clip}\left(r_t(\theta),\, 1-\varepsilon,\, 1+\varepsilon\right)\hat{A}_t\right)\right]
$$

where $\varepsilon$ is the **clip range** (0.1 in GOLDS).

#### How the clipping works:

| Scenario | $\hat{A}_t$ | Effect of clipping |
|---|---|---|
| Action was **good** ($\hat{A}_t > 0$) | Positive | Clipping caps $r_t$ at $1+\varepsilon$, preventing the policy from moving *too far* toward this action |
| Action was **bad** ($\hat{A}_t < 0$) | Negative | Clipping caps $r_t$ at $1-\varepsilon$, preventing the policy from moving *too far away* from this action |

The $\min$ operation is conservative: it always picks the lower (more pessimistic) objective, so the policy can never "cheat" by exploiting the surrogate.

In [ ]:
# --- PPO clipped objective visualization ---

def ppo_clip_objective(ratio, advantage, epsilon=0.1):
    """Compute the PPO clipped surrogate objective element-wise."""
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1.0 - epsilon, 1.0 + epsilon) * advantage
    return np.minimum(unclipped, clipped)


epsilon = 0.1  # GOLDS default clip_range
ratios = np.linspace(0.5, 1.8, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, adv, title in zip(
    axes,
    [1.0, -1.0],
    [r"Positive advantage ($\hat{A}_t = +1$)", r"Negative advantage ($\hat{A}_t = -1$)"],
):
    unclipped = ratios * adv
    clipped_obj = ppo_clip_objective(ratios, adv, epsilon)

    ax.plot(ratios, unclipped, "--", color="tab:blue", linewidth=2, label="Unclipped $r_t \\hat{A}_t$")
    ax.plot(ratios, clipped_obj, color="tab:red", linewidth=2.5, label="PPO clipped objective")
    ax.axvline(1.0, color="gray", linestyle=":", linewidth=1, label="$r_t = 1$ (no change)")
    ax.axvline(1.0 - epsilon, color="green", linestyle="--", linewidth=1, alpha=0.7, label=f"$1-\\varepsilon={1-epsilon}$")
    ax.axvline(1.0 + epsilon, color="green", linestyle="--", linewidth=1, alpha=0.7, label=f"$1+\\varepsilon={1+epsilon}$")
    ax.set_xlabel(r"Probability ratio $r_t(\theta)$")
    ax.set_ylabel("Objective")
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle(f"PPO Clipped Surrogate Objective ($\\varepsilon = {epsilon}$)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Reading the plots above:**

- **Left (positive advantage):** When the action was good, the unclipped objective rises linearly with $r_t$. The clipped version (red) flattens out once $r_t > 1 + \varepsilon$, removing the incentive to increase the action's probability beyond the trust region.
- **Right (negative advantage):** When the action was bad, the unclipped objective decreases as $r_t$ drops. The clipped version flattens once $r_t < 1 - \varepsilon$, preventing the policy from over-correcting.

---
## 3. Generalized Advantage Estimation (GAE)

### 3.1 The Bias-Variance Dilemma

We need to estimate the advantage $\hat{A}_t$ from data. There are two extremes:

- **1-step TD error** (low variance, high bias):
$$\delta_t^V = r_t + \gamma V(s_{t+1}) - V(s_t)$$
This uses the learned value function $V$ as a bootstrap, which introduces bias if $V$ is inaccurate.

- **Monte Carlo return** (high variance, zero bias):
$$\hat{A}_t^{\text{MC}} = \sum_{k=0}^{T-t} \gamma^k r_{t+k} - V(s_t)$$
This is unbiased but has high variance because it sums many noisy rewards.

### 3.2 GAE: The Best of Both Worlds

**Generalized Advantage Estimation** (Schulman et al., 2016) smoothly interpolates between these extremes using a parameter $\lambda \in [0, 1]$.

First, define the **k-step advantage estimator**:

$$
\hat{A}_t^{(k)} = \sum_{l=0}^{k-1} \gamma^l \delta_{t+l}^V = -V(s_t) + r_t + \gamma r_{t+1} + \cdots + \gamma^{k-1} r_{t+k-1} + \gamma^k V(s_{t+k})
$$

GAE is an **exponentially-weighted average** of all k-step estimators:

$$
\hat{A}_t^{\text{GAE}(\gamma, \lambda)} = (1 - \lambda)\left(\hat{A}_t^{(1)} + \lambda \hat{A}_t^{(2)} + \lambda^2 \hat{A}_t^{(3)} + \cdots\right)
$$

This simplifies to a beautiful recursive formula:

$$
\boxed{\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{T-t} (\gamma \lambda)^l \, \delta_{t+l}^V}
$$

Or equivalently, computed backwards:

$$
\hat{A}_t = \delta_t + \gamma \lambda \, \hat{A}_{t+1}
$$

### 3.3 Special Cases

| $\lambda$ | Estimator | Bias | Variance |
|---|---|---|---|
| $\lambda = 0$ | 1-step TD: $\hat{A}_t = \delta_t^V$ | High (bootstrap) | Low |
| $\lambda = 1$ | Monte Carlo: $\hat{A}_t = \sum \gamma^l \delta_{t+l}$ | Low | High |
| $\lambda = 0.95$ | **GOLDS default** — mostly MC but with some bias for stability | Medium-low | Medium |

GOLDS uses $\gamma = 0.99$ and $\lambda = 0.95$, which is the standard DeepMind/OpenAI setting.

In [ ]:
# --- GAE implementation from scratch ---

def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """
    Compute Generalized Advantage Estimation.

    Parameters
    ----------
    rewards : array of shape (T,)
        Rewards at each timestep.
    values : array of shape (T+1,)
        Value estimates. values[T] is V(s_T), the bootstrap value
        for the state *after* the last reward.
    gamma : float
        Discount factor.
    lam : float
        GAE lambda.

    Returns
    -------
    advantages : array of shape (T,)
    """
    T = len(rewards)
    advantages = np.zeros(T)

    # Work backwards (the recursive formula)
    last_gae = 0.0
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        last_gae = delta + gamma * lam * last_gae
        advantages[t] = last_gae

    return advantages


# --- Create a simple example trajectory ---
np.random.seed(42)
T = 20
rewards = np.random.randn(T) * 0.5 + 0.1  # noisy rewards
# Simulate a rough value function (increasing trend with noise)
true_trend = np.linspace(0, 2, T + 1)
values = true_trend + np.random.randn(T + 1) * 0.3

# Compute GAE for several lambda values
lambdas = [0.0, 0.5, 0.95, 1.0]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: show rewards and values
axes[0].bar(range(T), rewards, alpha=0.4, label="Rewards", color="tab:blue")
axes[0].plot(range(T + 1), values, "o-", markersize=4, label="$V(s_t)$ estimates", color="tab:orange")
axes[0].set_ylabel("Value")
axes[0].set_title("Example Trajectory: Rewards and Value Estimates")
axes[0].legend()

# Bottom: show GAE for different lambdas
colors = ["tab:red", "tab:purple", "tab:green", "tab:blue"]
for lam, color in zip(lambdas, colors):
    adv = compute_gae(rewards, values, gamma=0.99, lam=lam)
    axes[1].plot(range(T), adv, "o-", markersize=4, linewidth=2, color=color,
                 label=f"$\\lambda={lam}$" + (" (1-step TD)" if lam == 0 else "") + (" (MC)" if lam == 1 else ""))

axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_xlabel("Timestep $t$")
axes[1].set_ylabel("$\\hat{A}_t^{GAE}$")
axes[1].set_title("GAE Advantage Estimates for Different $\\lambda$")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Variance of advantage estimates across the trajectory:")
for lam in lambdas:
    adv = compute_gae(rewards, values, gamma=0.99, lam=lam)
    print(f"  lambda={lam:.2f}  ->  var={np.var(adv):.4f}")

**Interpreting the results above:**

- $\lambda = 0$ (red): The advantage at each step depends *only* on the immediate TD error. It is the least noisy but most biased — it trusts the value function $V$ completely.
- $\lambda = 1$ (blue): The advantage sums all future TD errors. It captures long-term information but has high variance (notice the wider swings).
- $\lambda = 0.95$ (green, the GOLDS default): A practical sweet spot. It looks similar to $\lambda=1$ but is slightly smoother.

---
## 4. The Full PPO Loss

PPO optimizes a **combined loss** with three terms:

$$
\boxed{L(\theta) = L^{\text{CLIP}}(\theta) - c_1 \, L^{\text{VF}}(\theta) + c_2 \, H\left[\pi_\theta\right](s_t)}
$$

Let's break down each term:

### 4.1 Policy Loss ($L^{\text{CLIP}}$)

This is the clipped surrogate we derived in Section 2:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)\,\hat{A}_t, \;\text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\,\hat{A}_t\right)\right]
$$

**GOLDS config:** `clip_range: 0.1` $\Rightarrow \varepsilon = 0.1$

### 4.2 Value Function Loss ($L^{\text{VF}}$)

We train the value head $V_\theta(s)$ alongside the policy. The value loss is typically a squared error:

$$
L^{\text{VF}}(\theta) = \mathbb{E}_t\left[\left(V_\theta(s_t) - V_t^{\text{target}}\right)^2\right]
$$

where $V_t^{\text{target}} = \hat{A}_t + V_{\theta_{\text{old}}}(s_t)$ (the GAE advantage plus the old value estimate).

**GOLDS config:** `vf_coef: 0.5` $\Rightarrow c_1 = 0.5$

The coefficient $c_1$ controls how much we weight value-function accuracy relative to policy improvement. Setting it to 0.5 is standard.

### 4.3 Entropy Bonus ($H[\pi_\theta]$)

To encourage exploration, we add the policy's entropy:

$$
H[\pi_\theta](s_t) = -\sum_a \pi_\theta(a \mid s_t) \log \pi_\theta(a \mid s_t)
$$

Higher entropy = more randomness = more exploration. The $+c_2$ sign means we *maximize* entropy (equivalently, we penalize the policy for becoming too deterministic too early).

**GOLDS config:** `ent_coef: 0.01` $\Rightarrow c_2 = 0.01$

### 4.4 Additional Training Parameters

| GOLDS Config | Symbol / Role | Value | Explanation |
|---|---|---|---|
| `learning_rate` | $\alpha$ | $2.5 \times 10^{-4}$ | Step size for Adam optimizer |
| `n_steps` | $T$ | 128 | Timesteps per rollout per environment before each update |
| `batch_size` | $M$ | 256 | Minibatch size for SGD within each epoch |
| `n_epochs` | $K$ | 4 | Number of passes over the rollout data per update |
| `max_grad_norm` | — | 0.5 | Gradient clipping (L2 norm) to prevent exploding gradients |
| `clip_range_vf` | $\varepsilon_V$ | `null` | Optional clipping for value function (disabled = trust the VF loss as-is) |

---
## 5. CnnPolicy Architecture

GOLDS uses the **Nature DQN** convolutional architecture (Mnih et al., 2015), which is the standard backbone for Atari-scale vision tasks.

### 5.1 Network Structure

The input is a **84 x 84 x 4** grayscale image (4 stacked frames, controlled by `frame_stack: 4` and `screen_size: 84` in the GOLDS config).

```
Input: 84 x 84 x 4
  │
  ├─ Conv2D(32 filters, 8x8, stride 4) + ReLU   →  20 x 20 x 32
  │
  ├─ Conv2D(64 filters, 4x4, stride 2) + ReLU   →  9 x 9 x 64
  │
  ├─ Conv2D(64 filters, 3x3, stride 1) + ReLU   →  7 x 7 x 64
  │
  ├─ Flatten                                      →  3136
  │
  ├─ Linear(3136, 512) + ReLU                     →  512  (shared features)
  │
  ├─── Policy head: Linear(512, n_actions)        →  n_actions
  │
  └─── Value head:  Linear(512, 1)                →  1
```

**Key insight:** The convolutional layers and the 512-unit dense layer are **shared** between the policy and value heads. This means the network learns visual features that are useful for *both* deciding what to do and estimating how good a state is.

In [ ]:
# --- Parameter count calculation for the Nature DQN CnnPolicy ---

def conv_output_size(input_size, kernel_size, stride, padding=0):
    """Calculate the spatial output size of a convolution."""
    return (input_size - kernel_size + 2 * padding) // stride + 1


def count_params(in_channels=4, input_size=84, n_actions=18):
    """Count parameters in the Nature DQN CnnPolicy.

    Default n_actions=18 is typical for many Atari games.
    """
    total = 0
    details = []

    # Conv1: in_channels -> 32, kernel 8x8, stride 4
    w = 32 * in_channels * 8 * 8
    b = 32
    out1 = conv_output_size(input_size, 8, 4)
    details.append(f"Conv1: {in_channels}x8x8x32 = {w:,} weights + {b} biases  (output: {out1}x{out1}x32)")
    total += w + b

    # Conv2: 32 -> 64, kernel 4x4, stride 2
    w = 64 * 32 * 4 * 4
    b = 64
    out2 = conv_output_size(out1, 4, 2)
    details.append(f"Conv2: 32x4x4x64 = {w:,} weights + {b} biases  (output: {out2}x{out2}x64)")
    total += w + b

    # Conv3: 64 -> 64, kernel 3x3, stride 1
    w = 64 * 64 * 3 * 3
    b = 64
    out3 = conv_output_size(out2, 3, 1)
    details.append(f"Conv3: 64x3x3x64 = {w:,} weights + {b} biases  (output: {out3}x{out3}x64)")
    total += w + b

    # Flatten
    flat_size = out3 * out3 * 64
    details.append(f"Flatten: {out3}x{out3}x64 = {flat_size:,}")

    # FC: flat_size -> 512
    w = flat_size * 512
    b = 512
    details.append(f"FC (shared): {flat_size}x512 = {w:,} weights + {b} biases")
    total += w + b

    # Policy head: 512 -> n_actions
    w = 512 * n_actions
    b = n_actions
    details.append(f"Policy head: 512x{n_actions} = {w:,} weights + {b} biases")
    total += w + b

    # Value head: 512 -> 1
    w = 512 * 1
    b = 1
    details.append(f"Value head: 512x1 = {w:,} weights + {b} biases")
    total += w + b

    return total, details


n_actions = 18  # typical Atari (e.g., up/down/left/right + fire combos)
total_params, details = count_params(n_actions=n_actions)

print("Nature DQN CnnPolicy — Parameter Count")
print("=" * 60)
for d in details:
    print(f"  {d}")
print("=" * 60)
print(f"  TOTAL: {total_params:,} parameters")
print(f"\n  (~{total_params / 1e6:.2f}M parameters, ~{total_params * 4 / 1024 / 1024:.1f} MB in float32)")

---
## 6. GOLDS Hyperparameter Map

The table below maps every field in `configs/defaults.yaml` to the mathematical notation used in this notebook.

### PPO Parameters

| Config Field | Math Symbol | Default | Section | Description |
|---|---|---|---|---|
| `ppo.learning_rate` | $\alpha$ | $2.5 \times 10^{-4}$ | §4.4 | Adam optimizer step size |
| `ppo.n_steps` | $T$ | 128 | §4.4 | Rollout length per environment |
| `ppo.batch_size` | $M$ | 256 | §4.4 | Minibatch size for SGD |
| `ppo.n_epochs` | $K$ | 4 | §4.4 | Epochs per PPO update |
| `ppo.gamma` | $\gamma$ | 0.99 | §1.1, §3 | Discount factor |
| `ppo.gae_lambda` | $\lambda$ | 0.95 | §3.2 | GAE bias-variance tradeoff |
| `ppo.clip_range` | $\varepsilon$ | 0.1 | §2.3 | PPO clipping parameter |
| `ppo.clip_range_vf` | $\varepsilon_V$ | `null` | §4.4 | Value function clipping (disabled) |
| `ppo.ent_coef` | $c_2$ | 0.01 | §4.3 | Entropy bonus coefficient |
| `ppo.vf_coef` | $c_1$ | 0.5 | §4.2 | Value function loss coefficient |
| `ppo.max_grad_norm` | — | 0.5 | §4.4 | Gradient clipping threshold |

### Environment Parameters

| Config Field | Default | Description |
|---|---|---|
| `environment.n_envs` | 8 | Parallel environments (total rollout = $n\_envs \times T$ steps) |
| `environment.frame_stack` | 4 | Consecutive frames stacked as CNN input channels |
| `environment.frame_skip` | 4 | Repeat each action for this many frames |
| `environment.screen_size` | 84 | Downscale frames to 84x84 |
| `environment.grayscale` | `true` | Convert RGB to grayscale |
| `environment.clip_reward` | `true` | Clip rewards to {-1, 0, +1} |
| `environment.terminal_on_life_loss` | `true` | Treat life loss as episode end |

### Training Parameters

| Config Field | Default | Description |
|---|---|---|
| `training.total_timesteps` | 10,000,000 | Total environment interactions |
| `training.eval_freq` | 50,000 | Evaluate every N steps |
| `training.eval_episodes` | 10 | Episodes per evaluation |
| `training.save_freq` | 100,000 | Save checkpoint every N steps |
| `training.device` | `auto` | CPU/GPU selection |

In [ ]:
# --- Load and display the actual GOLDS default configuration ---

from pathlib import Path

try:
    import yaml
except ImportError:
    print("PyYAML not installed. Install with: pip install pyyaml")
    yaml = None

if yaml is not None:
    # Try multiple paths so this works whether the notebook is run from
    # the repo root or from the notebooks/ directory.
    candidates = [
        Path("../configs/defaults.yaml"),
        Path("configs/defaults.yaml"),
        Path(__file__).parent.parent / "configs" / "defaults.yaml" if "__file__" in dir() else None,
    ]
    config_path = None
    for p in candidates:
        if p is not None and p.exists():
            config_path = p
            break

    if config_path is not None:
        with open(config_path) as f:
            defaults = yaml.safe_load(f)
        print(f"Loaded GOLDS defaults from: {config_path.resolve()}\n")
        print(yaml.dump(defaults, default_flow_style=False, sort_keys=False))
    else:
        print("Could not find configs/defaults.yaml — showing built-in defaults.")
        import json
        builtin = {
            "ppo": {
                "learning_rate": 2.5e-4, "n_steps": 128, "batch_size": 256,
                "n_epochs": 4, "gamma": 0.99, "gae_lambda": 0.95,
                "clip_range": 0.1, "clip_range_vf": None, "ent_coef": 0.01,
                "vf_coef": 0.5, "max_grad_norm": 0.5,
            },
            "environment": {
                "n_envs": 8, "frame_stack": 4, "frame_skip": 4,
                "screen_size": 84, "grayscale": True, "clip_reward": True,
                "terminal_on_life_loss": True, "use_subproc": True,
            },
            "training": {
                "total_timesteps": 10_000_000, "eval_freq": 50_000,
                "eval_episodes": 10, "save_freq": 100_000,
                "log_interval": 1, "seed": None, "device": "auto",
            },
        }
        print(json.dumps(builtin, indent=2))

In [ ]:
# --- Optionally load the GOLDS config loader itself ---

try:
    from golds.config.loader import ConfigLoader

    loader = ConfigLoader()
    print("GOLDS ConfigLoader imported successfully.")
    print(f"Default config keys: {list(loader._defaults.keys())}")
    print(f"\nPPO defaults from loader:")
    for k, v in loader._defaults.get("ppo", {}).items():
        print(f"  {k}: {v}")
except ImportError:
    print("GOLDS package not installed in this environment.")
    print("To install: cd /path/to/golds && pip install -e .")
except Exception as e:
    print(f"Could not load GOLDS config: {e}")

---
## Summary

Here is the full picture of how PPO works inside GOLDS, from data collection to gradient update:

1. **Collect rollouts:** Run the policy in 8 parallel environments (`n_envs`) for 128 steps each (`n_steps`), giving $8 \times 128 = 1{,}024$ transitions per update.

2. **Compute advantages:** Use GAE with $\gamma = 0.99$ and $\lambda = 0.95$ to estimate $\hat{A}_t$ for every transition.

3. **Optimize:** Shuffle the 1,024 transitions, split into minibatches of 256, and run 4 epochs of gradient descent on the combined loss:
$$L = L^{\text{CLIP}}(\varepsilon=0.1) - 0.5 \cdot L^{\text{VF}} + 0.01 \cdot H[\pi_\theta]$$
   Gradients are clipped to norm 0.5 (`max_grad_norm`).

4. **Repeat** for 10M total timesteps.

### Further Reading

- Schulman et al. (2017), "Proximal Policy Optimization Algorithms" — [arXiv:1707.06347](https://arxiv.org/abs/1707.06347)
- Schulman et al. (2016), "High-Dimensional Continuous Control Using Generalized Advantage Estimation" — [arXiv:1506.02438](https://arxiv.org/abs/1506.02438)
- Mnih et al. (2015), "Human-level control through deep reinforcement learning" — [Nature](https://www.nature.com/articles/nature14236)
- Sutton et al. (2000), "Policy Gradient Methods for Reinforcement Learning with Function Approximation" — [NeurIPS](https://papers.nips.cc/paper/1713-policy-gradient-methods-for-reinforcement-learning-with-function-approximation)